<a href="https://colab.research.google.com/github/umar-padela/SODA-policy/blob/dev_umar/data/pusht/pushT_labeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
so#@title installs
!pip install -q lerobot
!pip install -q mediapy lerobot
!pip install google-genai
!pip install "zarr<3.0.0"

In [ ]:
#@title global vars

SKILL_NAMES = {0: "STATIONARY", 1: "REPOSITION", 2: "LINEAR-PUSH", 3: "PIVOT-PUSH"}

# Color mapping (BGR format for OpenCV)
# Gray, Blue, Green, Orange/Gold
SKILL_COLORS = {
    0: (128, 128, 128), # Stationary
    1: (255, 0, 0),     # Reposition (Blue)
    2: (0, 255, 0),     # Linear Push (Green)
    3: (0, 165, 255)    # Pivot Push (Orange)
}

In [ ]:
#@title imports
import torch
from lerobot.datasets.lerobot_dataset import LeRobotDataset
import os
import cv2
from base64 import b64encode
from IPython.display import HTML, display
from lerobot.datasets.lerobot_dataset import LeRobotDataset
import torchvision.transforms.functional as TF
import numpy as np
import mediapy as media
from lerobot.datasets.lerobot_dataset import LeRobotDataset
import json
import mediapy as media
from google import genai
from google.genai import types
from google.colab import userdata
import time
from IPython.display import Markdown, Image, display
import math
import subprocess
import os
import gdown
import zarr
import numpy as np

# gemini setup
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
client = genai.Client()

In [ ]:
#@title load lerobot pushT dataset

# Load the official Push-T image/video dataset repo from Hugging Face
dataset = LeRobotDataset("lerobot/pusht")

print("Dataset loaded successfully!")
print("Total number of individual frames across all demos:", len(dataset))
print("Available data keys:", list(dataset.meta.features.keys()))

print("Original image tensor dimensions:", dataset[0]["observation.image"].shape)

In [ ]:
#@title helper functions

def extract_episode_frames(dataset, episode_idx: int, is_zarr: bool=False):
    """
    Extracts raw RGB numpy image frames and metadata for a specific
    episode from a LeRobotDataset instance.
    """
    if is_zarr:
      # dataset is the raw_data dictionary we created
      start_frame = 0 if episode_idx == 0 else int(dataset['episode_ends'][episode_idx - 1])
      end_frame = int(dataset['episode_ends'][episode_idx])
      fps = 10 # Standard for PushT

      # Slice directly from the numpy array
      raw_frames = dataset['images'][start_frame:end_frame]

      # Zarr frames are already uint8 [H, W, C], just convert to a list
      frames = [f.copy() for f in raw_frames]
    else:
      ep_meta = dataset.meta.episodes[episode_idx]
      start_frame = int(ep_meta["dataset_from_index"])
      end_frame = int(ep_meta["dataset_to_index"])
      fps = int(dataset.meta.fps)

      frames = []
      for frame_idx in range(start_frame, end_frame):
          sample = dataset[frame_idx]
          img_tensor = sample["observation.image"]

          # Convert from PyTorch [C, H, W] to standard uint8 [H, W, C] numpy format
          img_np = (img_tensor.permute(1, 2, 0).cpu().numpy() * 255).astype('uint8').copy()
          frames.append(img_np)

    return frames, fps

def burn_frame_number(frame, index):
    """Adds a high-contrast frame index to the top-left of a 96x96 frame."""
    display_frame = frame.copy()
    text = str(index)
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.22  # Small enough for 96x96
    thickness = 1
    text_x = 4
    text_y = 8

    # Red text
    cv2.putText(display_frame, text, (text_x, text_y), font, font_scale, (255, 0, 0), thickness, cv2.LINE_AA)
    return display_frame

def burn_state_overlay(frame, state_vector, label):
    """
    Adds Agent (A), Block (B), and Rotation (R) values to a 96x96 frame.
    Expects state_vector: [agent_x, agent_y, block_x, block_y, block_rot]
    """
    display_frame = frame.copy()
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.22  # Very small to fit 96x96
    thickness = 1

    # Round for display
    ax, ay, bx, by, rot = np.round(state_vector, 2)

    # Define text lines
    lines = [
        (f"A:{ax:.2f},{ay:.2f}", (0, 255, 0)),   # Green for Agent
        (f"B:{bx:.2f},{by:.2f}", (0, 255, 255)), # Yellow for Block
        (f"R:{rot:.2f}", (0, 0, 0))          # Cyan for Rotation
    ]

    # Draw lines on the right side of the frame
    for i, (text, color) in enumerate(lines):
        # Position them vertically starting at y=12
        cv2.putText(display_frame, text, (16, 8 + (i * 10)),
                    font, font_scale, color, thickness, cv2.LINE_AA)

    skill_color = SKILL_COLORS.get(label, (255, 255, 255)) # Default to White
    skill_name = SKILL_NAMES.get(label, "UNKNOWN")

    # 2. Draw a thick border (2 pixels)
    # Rectangle args: (image, top-left, bottom-right, color, thickness)
    cv2.rectangle(display_frame, (0, 0), (95, 95), skill_color, 2)

    # 5. Draw Skill Name in the Bottom Left
    # y=88 keeps it just above the bottom border
    cv2.putText(display_frame, f"S:{skill_name}", (4, 80),
                font, font_scale, skill_color, thickness, cv2.LINE_AA)

    return display_frame



def compile_frames_to_mp4(frames, fps: int, output_path: str = "temp_output.mp4", burn_in: bool = False, state=None, label=None):
    """
    Compiles a python list of uint8 RGB image arrays into an MP4 video file.
    Can optionally burn sequential frame numbers directly onto the pixels.
    """
    if not frames:
        raise ValueError("The frames list is empty.")

    height, width, _ = frames[0].shape
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    for relative_frame_idx, frame in enumerate(frames):
        # Create a copy so we don't accidentally mutate the underlying dataset arrays
        display_frame = frame.copy()

        # Apply visual frame numbers if requested
        if burn_in:
            display_frame = burn_frame_number(frame, relative_frame_idx)
        if state is not None:
            current_state = state[relative_frame_idx]
            current_label = label[relative_frame_idx]
            display_frame = burn_state_overlay(display_frame, current_state, current_label)
        # Crucial: Convert to BGR format right before feeding to OpenCV's writer
        bgr_frame = cv2.cvtColor(display_frame, cv2.COLOR_RGB2BGR)
        video_writer.write(bgr_frame.astype('uint8'))

    video_writer.release()
    return output_path

def compile_frames_to_grid(frames, output_path="video_grid.jpg"):
    """
    Tiles frames into a single large image grid.
    Optimized for 96x96 frames to prevent VLM downsampling.
    """
    if not frames:
        raise ValueError("The frames list is empty.")

    num_frames = len(frames)
    columns = columns = math.ceil(math.sqrt(num_frames))
    rows = math.ceil(num_frames / columns)
    h, w, c = frames[0].shape

    # Create a blank canvas (white background)
    grid_img = np.ones((rows * h, columns * w, c), dtype=np.uint8) * 255

    for idx, frame in enumerate(frames):
        # Burn in the frame number using the shared helper
        labeled_frame = burn_frame_number(frame, idx)

        r = idx // columns
        c_idx = idx % columns

        y_start, x_start = r * h, c_idx * w
        grid_img[y_start:y_start+h, x_start:x_start+w] = labeled_frame

    # Convert to BGR for OpenCV saving
    cv2.imwrite(output_path, cv2.cvtColor(grid_img, cv2.COLOR_RGB2BGR))
    return output_path

def play_episode(dataset, episode_idx: int, is_zarr: bool = False, burn_in: bool = True, height: int = 448, width: int = 448):
    """
    Plays an episode from either a LeRobotDataset or a Zarr dictionary.
    """
    temp_playback_path = f"playback_episode_{episode_idx}.mp4"

    # 1. Use the unified extractor with the flag
    # If is_zarr=True, 'dataset' should be your raw_data dictionary
    frames, fps = extract_episode_frames(dataset, episode_idx, is_zarr=is_zarr)

    # 2. Compile the video (This function is already compatible with both)
    state = None
    label = None
    if is_zarr:
      start_idx = 0 if episode_idx == 0 else int(dataset['episode_ends'][episode_idx - 1])
      end_idx = int(dataset['episode_ends'][episode_idx])
      state = dataset['state'][start_idx:end_idx]
      label=dataset['option_id'][start_idx:end_idx]

    compile_frames_to_mp4(frames, fps, output_path=temp_playback_path, burn_in=burn_in, state=state, label=label)

    dataset_type = "Zarr" if is_zarr else "LeRobot"
    print(f"Playing {dataset_type} Episode {episode_idx} ({len(frames)} frames)...")

    # 3. Render in Colab
    video_to_play = media.read_video(temp_playback_path)
    media.show_video(video_to_play, height=height, width=width)

    # Clean up
    if os.path.exists(temp_playback_path):
        os.remove(temp_playback_path)


In [ ]:
#@title play episode


# 2. Extract boundaries for Episode 0 from your metadata stats
episode_idx = 4 #@param
play_episode(dataset, episode_idx, burn_in=False)

In [ ]:
#@title play episode with frame numbers burnt in for initial ground truth labels
episode_idx = 111 #@param
play_episode(dataset, episode_idx, burn_in=True)

In [ ]:
#@title label video with gemini

def slow_down_video(input_path, output_path, speed_factor=0.5):
    """
    Slows down the video physically so the VLM can process more detail.
    speed_factor 0.5 = 2x slower.
    """
    # pts = 1/speed_factor. 1/0.5 = 2.0
    setpts = 1 / speed_factor
    cmd = [
        'ffmpeg', '-i', input_path,
        '-filter:v', f"setpts={setpts}*PTS",
        '-y', output_path
    ]
    subprocess.run(cmd, check=True)




def label_video_gemini(dataset, episode_idx: int):
    """
    Full SODA pipeline orchestrator: Extracts frames, compiles video,
    uploads to Gemini 3 Flash, and retrieves structured JSON skill boundaries.
    """
    temp_video_path = f"soda_episode_{episode_idx}.mp4"


    # 1. Fetch frames using Module 1
    print(f"\n--- SODA Labeling: Processing Episode {episode_idx} ---")
    frames, fps = extract_episode_frames(dataset, episode_idx)
    print(f"Extracted {len(frames)} frames from LeRobot dataset at {fps} FPS.")

    # 2. Compile video locally using Module 2
    compile_frames_to_mp4(frames, fps, output_path=temp_video_path, burn_in=True)
    print(f"Successfully compiled temporary video file: {temp_video_path}")

    # slow_video_path = f"slow_soda_{episode_idx}.mp4"
    # slow_down_video(temp_video_path, slow_video_path, speed_factor=0.75)
    # print(f"Successfully slowed down temporary video file: {slow_video_path}")
    # temp_video_path = slow_video_path

    try:
        # 3. Upload to Gemini Cloud File Architecture
        print("Uploading file asset to Gemini File API...")
        cloud_video = client.files.upload(file=temp_video_path)

        while cloud_video.state.name == "PROCESSING":
            print(".", end="")
            time.sleep(4)
            cloud_video = client.files.get(name=cloud_video.name)

        if cloud_video.state.name == "FAILED":
            raise ValueError("Cloud file tracking processing failed.")

        print("\nAsset state is ACTIVE. Requesting SODA segmentation analysis...")

        # Our localized, jargon-free video labeling prompt
        soda_prompt = """
        Role: Robotics Data Specialist
        Task: Identify and segment distinct skills within a robot manipulation video.

        ## Video Context ##
        - the robot is using it's blue end-effector to move the gray T-block into the green T-shape target. your job is to watch the video, understand the high-level strategy, and decompose it into a sequence of three discrete skills, REPOSITION, LINEAR-PUSH, and PIVOT-PUSH.

        **Environment Elements:**
        - **Blue Circle:** The robot's end-effector (the mover controlled by the agent).
        - **Gray T-Block:** A moveable object that shifts only when physically pushed by the blue circle.
        - **Green T-Shape:** A static target zone printed on the background table surface.

        **Skill Definitions:**
        Your primary objective is to identify the precise frames where the strategy transitions between skills.
        1. **REPOSITION:** Starts at frame 0. Also starts whenever the gray T-block stops moving.
        2. **LINEAR-PUSH:** Starts the exact frame that the gray T-block **begins moving**. Classify as LINEAR if the subsequent frames show the block sliding in a straight line without significant rotation.
        3. **PIVOT-PUSH:** Starts the exact frame the gray T-block **begins moving**. Classify as PIVOT if the subsequent frames show the block rotating (spinning) and translating simultaneously.
        Remember, skills can be as short as a few frames, so analyze the motion closely.

        **Frame Number Tracking:**
        Read the burned-in red frame counter in the top-left corner directly. Do not estimate timestamps.

        **Output Format Requirements:**
        You must output exactly two distinct sections separated by a horizontal rule (`---`). Mirror the exact structural format of the Golden Example below. Do not add introductory remarks, markdown code block wrappers around the entire response, or trailing commentary.

        ** most important rule **
        any frame where the gray-T block moves by even a single pixel must be classified as either a LINEAR-PUSH or PIVOT-PUSH, it cannot be classified as a reposition. any frame where the gray T-block is stationary and the blue circle is moving must be classified as a REPOSITION, not a PUSH.

        ## processing workflow ##
        first, watch the video and output a chain of thought reasoning of the sequence of REPOSITION, LINEAR-PUSH, and PIVOT-PUSH employed in the demonstration.
        second, read through that chain of thought reasoning, and identify the precise frame number for each transition.
        third, compare your frame number assignmetns to the **most important rule** that was defined above, and ensure that your assignments are consistent with the most important rule.
        fourth, update the frame number assignments if they are not consistent with the most important rule.
        fifth, generate the JSON array.

        ### Golden Example Output:

        Chain of thought reasoning:
        the end-effector starts in free space far away to the right of the gray-T. it begins with a REPOSITION, to get in contact with the left side of the gray-T. once in contact, it uses a PIVOT-PUSH to rotate and push the gray-T towards the green target.
        afterwards, it does another REPOSITION to get to the top of the gray-T. then it does a PIVOT-PUSH to rotate and push the gray-T down towards the green target. then it does another REPOSITION to get the left side of the T.
        then, it uses a LINEAR-PUSH to push the gray-T towards the green target. then it does another REPOSITION to get to the top of the gray-T. finally, it does a LINEAR-PUSH to fully move the gray-T into the green target.

        REPOSITION: 0-41
        PIVOT-PUSH: 41-55
        REPOSITION: 55-94
        PIVOT-PUSH: 94-105
        REPOSITION: 105-129
        PIVOT-PUSH: 129-134
        REPOSITION: 134-149
        LINEAR-PUSH: 149-158

        Self-correction check: I noticed that I ended the first PIVOT-PUSH at frame 55, but it really continues all the way until frame 68, when the block finally comes to rest. updating frames.
        ---
        [
          {"skill": "REPOSITION", "start": 0, "end": 41},
          {"skill": "PIVOT-PUSH", "start": 41, "end": 68},
          {"skill": "REPOSITION", "start": 68, "end": 94},
          {"skill": "PIVOT-PUSH", "start": 94, "end": 105},
          {"skill": "REPOSITION", "start": 105, "end": 129},
          {"skill": "PIVOT-PUSH", "start": 129, "end": 134},
          {"skill": "REPOSITION", "start": 134, "end": 149},
          {"skill": "LINEAR-PUSH", "start": 149, "end": 158}
        ]
        """


        response = client.models.generate_content(
          #model="gemini-3.1-flash-lite",
          model="gemini-3.1-pro-preview",
          #model="gemini-robotics-er-1.6-preview",
          contents=types.Content(
              parts=[
                  # Manually construct the Part to include video_metadata
                  types.Part(
                      file_data=types.FileData(
                          file_uri=cloud_video.uri,
                          mime_type="video/mp4" # Tells the API it's an MP4
                      ),
                      # This forces Gemini to look at 10 frames per second
                      video_metadata=types.VideoMetadata(fps=10)
                  ),
                  # Your text prompt must also be wrapped in a text Part
                  types.Part(text=soda_prompt)
              ]
          ),
          config=types.GenerateContentConfig(
              temperature=0.0,
              thinking_config=types.ThinkingConfig(
                  include_thoughts=True,
                  thinking_budget=4000
              ),
          )
      )



        print("\n--- Model Reasoning & JSON Boundaries ---")
        display(Markdown(response.text))

        print("Prompt tokens:",response.usage_metadata.prompt_token_count)
        print("Thoughts tokens:",response.usage_metadata.thoughts_token_count)
        print("Output tokens:",response.usage_metadata.candidates_token_count)
        print("Total tokens:",response.usage_metadata.total_token_count)



    finally:
        # 5. Clean up tracking files in BOTH cloud bucket and local environment
        if 'cloud_video' in locals():
            client.files.delete(name=cloud_video.name)
            print(f"Removed remote asset from Gemini Storage Bucket.")

        if os.path.exists(temp_video_path):
            os.remove(temp_video_path)
            print(f"Removed temporary local file: {temp_video_path}")

    return response.text

episode_idx = 1 #@param
play_episode(dataset, episode_idx, burn_in=True)
label_video_gemini(dataset, episode_idx)

## findings

it seems like the VLM struggles to identify the exact high level reasonign and discrete skills for this task, I think it will do better for something like robomimic SQUARE for which there is more semantic meaning.

it seems like lerobot does not include the block coordinates in the dataset, which is needed for labeling, so try downloading from columbia site here

https://diffusion-policy.cs.columbia.edu/data/training/

In [ ]:
#@title download columbia pushT dataset directly

# 1. Download the raw file
dataset_path = "pusht_cchi_v7_replay.zarr.zip"
if not os.path.isfile(dataset_path):
    # This is the original file ID from the Columbia Diffusion Policy paper
    id = "1KY1InLurpMvJDRb14L9NlXT_fEsCvVUq&confirm=t"
    gdown.download(id=id, output=dataset_path, quiet=False)

# 2. Open the Zarr store
# 'r' means read-only mode
store = zarr.storage.ZipStore(dataset_path, mode='r')
z = zarr.open(store=store, mode='r')

# 3. Pull everything into a flat format
# In Zarr, you access data like a dictionary: z['folder']['dataset']
dataset_zarr = {
    'images': z['data']['img'][:],          # (Total_Frames, 96, 96, 3)
    'state': z['data']['state'][:],        # (Total_Frames, 5) -> [AgentX, AgentY, BlockX, BlockY, BlockRot]
    'action': z['data']['action'][:],      # (Total_Frames, 2)
    'episode_ends': z['meta']['episode_ends'][:] # Indices where each episode finishes
}

# Convert index 4 (Rotation) from Radians to Degrees permanently for this dict
dataset_zarr['state'][:, 4] = np.rad2deg(dataset_zarr['state'][:, 4])

print(f"Total frames loaded: {dataset_zarr['state'].shape[0]}")
print(f"Number of episodes: {len(dataset_zarr['episode_ends'])}")

In [ ]:
#@title label pushT dataset using exact state data
def label_skills(raw_data):
    """
    0: STATIONARY, 1: REPOSITION, 2: LINEAR-PUSH, 3: PIVOT-PUSH
    """
    # --- CONSTANTS ---
    GRIPPER_MOVE_THRESHOLD = 0.01
    BLOCK_MOVE_THRESHOLD = 0.2
    BLOCK_ROT_THRESHOLD = 0.1
    PIVOT_AVG_THRESHOLD = 1.0
    # -----------------

    states = raw_data['state']
    episode_ends = raw_data['episode_ends']
    num_frames = states.shape[0]
    labels = np.zeros(num_frames, dtype=int)

    start_idx = 0
    for ep_end in episode_ends:
        ep_states = states[start_idx:ep_end]
        n_ep_frames = len(ep_states)
        ep_labels = np.zeros(n_ep_frames, dtype=int)

        # --- 1. PRE-CALCULATE VELOCITIES FOR THIS EPISODE ---
        # Gripper translation
        g_vels = np.linalg.norm(ep_states[1:, 0:2] - ep_states[:-1, 0:2], axis=1)

        # Block translation
        b_vels = np.linalg.norm(ep_states[1:, 2:4] - ep_states[:-1, 2:4], axis=1)

        # Block rotation with 360 wrap-around
        raw_rot_diffs = np.abs(ep_states[1:, 4] - ep_states[:-1, 4])
        b_rot_vels = np.minimum(raw_rot_diffs, 360 - raw_rot_diffs)

        # --- 2. FIRST PASS: IDENTIFY STATIONARY vs REPO vs PUSHING ---
        for i in range(n_ep_frames - 1):
            g_moving = g_vels[i] > GRIPPER_MOVE_THRESHOLD
            # Block is moving if it translates OR rotates
            b_moving = (b_vels[i] > BLOCK_MOVE_THRESHOLD) or (b_rot_vels[i] > BLOCK_ROT_THRESHOLD)

            if g_moving:
                if not b_moving:
                    ep_labels[i] = 1 # REPOSITION
                else:
                    ep_labels[i] = -1 # PUSHING (Pending classification)
            else:
                ep_labels[i] = 0 # STATIONARY

        # --- 3. SECOND PASS: CLASSIFY PUSHING SEGMENTS ---
        i = 0
        while i < n_ep_frames:
            if ep_labels[i] == -1:
                seg_start = i
                while i < n_ep_frames and ep_labels[i] == -1:
                    i += 1
                seg_end = i

                # Check average rotation rate of this specific segment
                # We slice b_rot_vels from seg_start up to seg_end-1
                if seg_end > seg_start + 1:
                    avg_rot = np.mean(b_rot_vels[seg_start:seg_end-1])
                    skill = 3 if avg_rot >= PIVOT_AVG_THRESHOLD else 2
                else:
                    skill = 2 # Default for 1-frame "blips"

                ep_labels[seg_start:seg_end] = skill
            else:
                i += 1

        # Final cleanup
        ep_labels[-1] = ep_labels[-2]
        labels[start_idx:ep_end] = ep_labels
        start_idx = ep_end

    return labels



labels_array = label_skills(dataset_zarr)
# Add it to the dictionary
dataset_zarr['option_id'] = labels_array

# Map for printing
skill_names = {0: "STATIONARY", 1: "REPOSITION", 2: "LINEAR-PUSH", 3: "PIVOT-PUSH"}

In [ ]:
#@title label statistics
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

def visualize_labels(labels, episode_ends):
    """
    Calculates aggregate stats, plots length distributions,
    and identifies episodes with the shortest segments.
    """

    # 1. Aggregate Counts
    total_points = len(labels)
    counts = Counter(labels)

    print("--- GLOBAL AGGREGATES ---")
    print(f"{'Skill':<15} | {'Points':<10} | {'Percentage':<10}")
    print("-" * 40)
    for s_id, name in skill_names.items():
        count = counts.get(s_id, 0)
        percent = (count / total_points) * 100
        print(f"{name:<15} | {count:<10} | {percent:>8.2f}%")
    print("-" * 40)
    print(f"{'TOTAL':<15} | {total_points:<10} | 100.00%\n")

    # 2. Segment Length & Short Segment Analysis
    lengths = {0: [], 1: [], 2: [], 3: []}
    starts = {0: [], 1: [], 2: [], 3: []} # Added to track if segment starts at frame 0
    short_segment_map = {1: [], 2: [], 3: []}

    start_idx = 0
    for ep_idx, ep_end in enumerate(episode_ends):
        ep_labels = labels[start_idx:ep_end]

        if len(ep_labels) > 0:
            current_skill = ep_labels[0]
            current_len = 0
            current_segment_start = 0

            for i, l in enumerate(ep_labels):
                if l == current_skill:
                    current_len += 1
                else:
                    lengths[current_skill].append(current_len)
                    starts[current_skill].append(current_segment_start) # Keep track of start
                    if current_skill in short_segment_map:
                        short_segment_map[current_skill].append((ep_idx, current_len, current_segment_start, i - 1))

                    current_skill = l
                    current_len = 1
                    current_segment_start = i

            # Last segment of the episode
            lengths[current_skill].append(current_len)
            starts[current_skill].append(current_segment_start) # Keep track of start
            if current_skill in short_segment_map:
                short_segment_map[current_skill].append((ep_idx, current_len, current_segment_start, len(ep_labels) - 1))

        start_idx = ep_end

    # 3. Statistics Table (Original - Unchanged)
    print("--- SEGMENT LENGTH STATISTICS (Frames) ---")
    print(f"{'Skill':<15} | {'Mean':<6} | {'Std':<6} | {'Min':<5} | {'Max':<5}")
    print("-" * 50)
    for s_id, name in skill_names.items():
        data = lengths[s_id]
        if data:
            mu, std = np.mean(data), np.std(data)
            mn, mx = np.min(data), np.max(data)
            print(f"{name:<15} | {mu:>6.1f} | {std:>6.1f} | {mn:>5} | {mx:>5}")
        else:
            print(f"{name:<15} | N/A")
    print("-" * 50 + "\n")

    # 3b. Short Segment Distribution Table (New)
    print("--- SHORT SEGMENT ANALYSIS ---")
    print(f"{'Skill':<15} | {'<2 fr %':<8} | {'<3 fr %':<8} | {'<4 fr %':<8} | {'Mid-Ep <4 %'}")
    print("-" * 65)
    for s_id, name in skill_names.items():
        l_data = np.array(lengths[s_id])
        s_data = np.array(starts[s_id])
        if len(l_data) > 0:
            p2 = (np.sum(l_data < 2) / len(l_data)) * 100
            p3 = (np.sum(l_data < 3) / len(l_data)) * 100

            mask_lt4 = l_data < 4
            p4 = (np.sum(mask_lt4) / len(l_data)) * 100

            # Out of segments < 4, how many did not start at frame 0
            short_not_at_0 = np.sum((mask_lt4) & (s_data != 0))
            total_short = np.sum(mask_lt4)
            p4_mid = (short_not_at_0 / total_short * 100) if total_short > 0 else 0.0

            print(f"{name:<15} | {p2:>7.1f}% | {p3:>7.1f}% | {p4:>7.1f}% | {p4_mid:>10.1f}%")
        else:
            print(f"{name:<15} | N/A")
    print("-" * 65 + "\n")

    # 4. Short Segment Episode Reporting
    print("--- EPISODES WITH SHORTEST SEGMENTS ---")
    for s_id in [1, 2, 3]:
        name = skill_names[s_id]
        sorted_episodes = sorted(short_segment_map[s_id], key=lambda x: x[1])

        unique_report = []
        seen_eps = set()
        for ep_i, length, f_start, f_end in sorted_episodes:
            if ep_i not in seen_eps:
                unique_report.append(f"Ep {ep_i} frame {f_start}-{f_end} (len {length})")
                seen_eps.add(ep_i)
            if len(unique_report) >= 3:
                break

        print(f"{name:<12}: " + ", ".join(unique_report))
    print("")

    # 4b. Longest Segment Episode Reporting (New)
    print("--- EPISODES WITH LONGEST SEGMENTS ---")
    for s_id in [1, 2, 3]:
        name = skill_names[s_id]
        # Sort by length descending
        sorted_episodes = sorted(short_segment_map[s_id], key=lambda x: x[1], reverse=True)

        unique_report = []
        seen_eps = set()
        for ep_i, length, f_start, f_end in sorted_episodes:
            if ep_i not in seen_eps:
                unique_report.append(f"Ep {ep_i} frame {f_start}-{f_end} (len {length})")
                seen_eps.add(ep_i)
            if len(unique_report) >= 3:
                break

        print(f"{name:<12}: " + ", ".join(unique_report))

    # 5. Histograms
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes = axes.flatten()
    colors = ['gray', 'blue', 'green', 'orange']

    for i, (s_id, name) in enumerate(skill_names.items()):
        data = lengths[s_id]
        if data:
            axes[i].hist(data, bins=30, color=colors[i], edgecolor='black', alpha=0.7)
            axes[i].set_title(f"Length Distribution: {name}")
            axes[i].set_xlabel("Frames")
            axes[i].set_ylabel("Frequency")
        else:
            axes[i].set_title(f"{name} (No Data)")

    plt.tight_layout()
    plt.show()

    target_segment = next((seg for seg in short_segment_map[1] if seg[1] == 3), "No 3-frame segment found")
    print(f"Target 3-frame segment: {target_segment}")

# Run the visualization
visualize_labels(dataset_zarr['option_id'], dataset_zarr['episode_ends'])

In [ ]:
dataset_zarr.keys()

In [ ]:
#@title play episode from zarr dataset
episode_idx = 100 #@param
play_episode(dataset_zarr, episode_idx, is_zarr=True)

In [ ]:
import os
import shutil
import numpy as np
import zarr

# 1. Double check we are on a V2 lifecycle
assert zarr.__version__.startswith('2.'), f"Please run !pip install 'zarr<3.0.0' and restart runtime. Current version: {zarr.__version__}"

# 2. CRITICAL: Convert block rotation (index 4) back to radians for neural net scaling
dataset_zarr['state'][:, 4] = np.deg2rad(dataset_zarr['state'][:, 4])

# 3. Use the standard Zarr V2 DirectoryStore
local_zarr_path = '/content/pushT_options_labeled.zarr'
store_out = zarr.DirectoryStore(local_zarr_path)
root_out = zarr.group(store=store_out, overwrite=True)

# 4. Recreate the exact hierarchy expected by the Diffusion Policy repo
data_group = root_out.create_group('data')
meta_group = root_out.create_group('meta')

# 5. Dump arrays using the standard .array() method
data_group.array('img', dataset_zarr['images'], chunks=(100, 96, 96, 3), dtype='uint8')
data_group.array('state', dataset_zarr['state'], chunks=(1000, 5), dtype='float32')
data_group.array('action', dataset_zarr['action'], chunks=(1000, 2), dtype='float32')
meta_group.array('episode_ends', dataset_zarr['episode_ends'], dtype='int32')

# 6. Add your newly minted option_id key alongside the other trajectory keys
data_group.array('option_id', dataset_zarr['option_id'], chunks=(1000,), dtype='int32')

# 7. Compress into a single zip archive for fast download
shutil.make_archive('/content/pushT_options_labeled', 'zip', local_zarr_path)
print("SUCCESS: Your Zarr V2 dataset is ready for local download!")

In [ ]:
from google.colab import files

# This will trigger a native browser download for the zip file
files.download('/content/pushT_options_labeled.zip')